In [6]:
# ============================================================
# FOOTBALL MARKET VALUE STACKED MODEL - ROBUST VERSION
# ============================================================
# This script:
# 1. Loads EA, Transfermarkt and FBref data
# 2. Cleans names and date columns
# 3. Creates engineered features, including FBref per90 metrics
# 4. Builds a stacked regression pipeline
# 5. Evaluates performance
# 6. Saves predictions, metrics, and SHAP feature importance
#
# Main target:
#   average(EA value, Transfermarkt value) when both are available
#
# Notes:
# - This version avoids the "empty test_df" bug
# - It is designed to be practical and easier to maintain
# ============================================================

import os
import re
import json
import warnings
import unicodedata
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# Optional packages
# ------------------------------------------------------------
USE_LIGHTGBM = True
USE_SHAP = True

try:
    from lightgbm import LGBMRegressor
except Exception:
    USE_LIGHTGBM = False

try:
    import shap
except Exception:
    USE_SHAP = False

from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ============================================================
# CONFIG
# ============================================================

EA_PATH = "../player_stats_cleaned.csv"
TM_PATH = "../transfermarkt_merged_players_with_valuation.csv"
FB_PATH = "../Fbref_Final_Data.csv"

OUTPUT_DIR = "../model_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

RANDOM_STATE = 42
N_SPLITS = 5

# ============================================================
# HELPER FUNCTIONS
# ============================================================

def normalize_name(name):
    """
    Make player names easier to match across datasets.
    """
    if pd.isna(name):
        return np.nan

    name = str(name)
    name = unicodedata.normalize("NFKD", name).encode("ascii", "ignore").decode("ascii")
    name = name.lower().strip()
    name = re.sub(r"[^a-z0-9\s]", "", name)
    name = re.sub(r"\s+", " ", name).strip()
    return name


def safe_datetime(series):
    """
    Convert a series to datetime safely.
    """
    return pd.to_datetime(series, errors="coerce")


def regression_metrics(y_true, y_pred):
    """
    Calculate evaluation metrics for regression.
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    pct_error = np.where(y_true != 0, (y_pred - y_true) / y_true, np.nan)
    abs_pct_error = np.abs(pct_error)

    acc10 = np.nanmean(abs_pct_error <= 0.10) * 100
    acc20 = np.nanmean(abs_pct_error <= 0.20) * 100
    mpe = np.nanmean(pct_error) * 100
    mape = np.nanmean(abs_pct_error) * 100

    return {
        "RMSE": float(rmse),
        "MAE": float(mae),
        "R2": float(r2),
        "Accuracy@10%": float(acc10),
        "Accuracy@20%": float(acc20),
        "Mean Percentage Error": float(mpe),
        "MAPE": float(mape),
    }


def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)


def infer_position_group(pos):
    """
    Map detailed positions into broad groups.
    """
    if pd.isna(pos):
        return "UNK"

    pos = str(pos).upper()

    if "GK" in pos:
        return "GK"

    defensive_tokens = ["CB", "LB", "RB", "LWB", "RWB", "DEF", "DF"]
    midfield_tokens = ["DM", "CM", "AM", "LM", "RM", "MID", "MF", "CAM", "CDM"]
    forward_tokens = ["ST", "CF", "LW", "RW", "FW", "ATT", "WF"]

    if any(t in pos for t in defensive_tokens):
        return "DEF"
    if any(t in pos for t in midfield_tokens):
        return "MID"
    if any(t in pos for t in forward_tokens):
        return "FWD"

    return "UNK"


def add_position_group(df):
    """
    Create a broad position group from whichever position column exists.
    """
    df = df.copy()

    if "positions" in df.columns:
        df["position_group"] = df["positions"].apply(infer_position_group)
    elif "Pos" in df.columns:
        df["position_group"] = df["Pos"].apply(infer_position_group)
    elif "position" in df.columns:
        df["position_group"] = df["position"].apply(infer_position_group)
    elif "sub_position" in df.columns:
        df["position_group"] = df["sub_position"].apply(infer_position_group)
    else:
        df["position_group"] = "UNK"

    return df


def add_fbref_per90(df):
    """
    Create some useful per90 metrics for FBref.
    """
    df = df.copy()

    # If 90s is missing, derive from minutes if possible
    if "90s" not in df.columns and "Min" in df.columns:
        df["90s"] = pd.to_numeric(df["Min"], errors="coerce") / 90.0

    count_cols = ["Gls", "Ast", "G+A", "G-PK", "PK", "PKatt", "Sh", "SoT", "Int", "TklW"]

    for col in count_cols:
        if col in df.columns and "90s" in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
            df[f"{col}_per90_calc"] = np.where(df["90s"] > 0, df[col] / df["90s"], np.nan)

    if {"Gls", "Ast", "90s"}.issubset(df.columns):
        df["goal_contrib"] = df["Gls"] + df["Ast"]
        df["goal_contrib_per90"] = np.where(df["90s"] > 0, df["goal_contrib"] / df["90s"], np.nan)

    if {"Starts", "MP"}.issubset(df.columns):
        df["start_ratio"] = np.where(df["MP"] > 0, df["Starts"] / df["MP"], np.nan)

    if {"Min", "MP"}.issubset(df.columns):
        df["minutes_per_match"] = np.where(df["MP"] > 0, df["Min"] / df["MP"], np.nan)

    return df


def make_design_matrices(train_df, test_df, feature_cols):
    """
    One-hot encode train and test together so columns always align.
    Supports empty test_df.
    """
    if len(test_df) > 0:
        combined = pd.concat([train_df[feature_cols], test_df[feature_cols]], axis=0)
        combined = pd.get_dummies(combined, dummy_na=True)

        x_train = combined.iloc[:len(train_df)].copy()
        x_test = combined.iloc[len(train_df):].copy()
    else:
        x_train = pd.get_dummies(train_df[feature_cols], dummy_na=True)
        x_test = pd.DataFrame(columns=x_train.columns)

    for col in x_train.columns:
        x_train[col] = pd.to_numeric(x_train[col], errors="coerce")

    if len(x_test) > 0:
        for col in x_test.columns:
            x_test[col] = pd.to_numeric(x_test[col], errors="coerce")

    return x_train, x_test


def fit_imputer(x_train, x_test=None):
    """
    Fit a median imputer to training data and transform train/test.
    """
    imputer = SimpleImputer(strategy="median")
    x_train_i = pd.DataFrame(
        imputer.fit_transform(x_train),
        columns=x_train.columns,
        index=x_train.index
    )

    if x_test is not None and len(x_test) > 0:
        x_test_i = pd.DataFrame(
            imputer.transform(x_test),
            columns=x_test.columns,
            index=x_test.index
        )
    else:
        x_test_i = pd.DataFrame(columns=x_train.columns)

    return x_train_i, x_test_i, imputer


def build_model():
    """
    Use LightGBM if available, else fall back to ExtraTrees.
    """
    if USE_LIGHTGBM:
        return LGBMRegressor(
            n_estimators=500,
            learning_rate=0.03,
            max_depth=6,
            num_leaves=31,
            subsample=0.9,
            colsample_bytree=0.9,
            random_state=RANDOM_STATE,
            verbosity=-1
        )
    else:
        return ExtraTreesRegressor(
            n_estimators=500,
            max_depth=14,
            min_samples_leaf=2,
            random_state=RANDOM_STATE,
            n_jobs=-1
        )


def generate_oof_preds(train_df, test_df, feature_cols, target_col, label):
    """
    Generate proper out-of-fold predictions for stacking.
    Also supports empty test_df safely.
    """
    x_train, x_test = make_design_matrices(train_df, test_df, feature_cols)
    y_train = train_df[target_col].values

    oof = np.zeros(len(train_df))
    test_preds = np.array([]) if len(test_df) == 0 else np.zeros(len(test_df))

    kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

    for fold, (tr_idx, va_idx) in enumerate(kf.split(x_train), start=1):
        x_tr = x_train.iloc[tr_idx]
        x_va = x_train.iloc[va_idx]
        y_tr = y_train[tr_idx]

        x_tr_i, x_va_i, imputer = fit_imputer(x_tr, x_va)

        model = build_model()
        model.fit(x_tr_i, y_tr)

        oof[va_idx] = model.predict(x_va_i)

        if len(test_df) > 0:
            x_test_i = pd.DataFrame(
                imputer.transform(x_test),
                columns=x_test.columns,
                index=x_test.index
            )
            test_preds += model.predict(x_test_i) / N_SPLITS

        print(f"[{label}] fold {fold}/{N_SPLITS} complete")

    # Final model trained on all training data
    x_train_i, x_test_i, final_imputer = fit_imputer(x_train, x_test if len(test_df) > 0 else None)
    final_model = build_model()
    final_model.fit(x_train_i, y_train)

    return {
        "oof": oof,
        "test_preds": test_preds,
        "model": final_model,
        "imputer": final_imputer,
        "x_train": x_train_i,
        "x_test": x_test_i,
        "feature_names": list(x_train.columns)
    }


def score_new_data(model, imputer, train_feature_names, new_df, raw_feature_cols):
    """
    Score a new dataframe using the same one-hot layout and imputer
    from training.
    """
    x_new = pd.get_dummies(new_df[raw_feature_cols], dummy_na=True)

    for col in train_feature_names:
        if col not in x_new.columns:
            x_new[col] = 0

    extra_cols = [c for c in x_new.columns if c not in train_feature_names]
    if extra_cols:
        x_new = x_new.drop(columns=extra_cols)

    x_new = x_new[train_feature_names]
    x_new = x_new.apply(pd.to_numeric, errors="coerce")

    x_new_i = pd.DataFrame(
        imputer.transform(x_new),
        columns=x_new.columns,
        index=x_new.index
    )

    return model.predict(x_new_i)


def get_existing_columns(df, cols):
    return [c for c in cols if c in df.columns]


# ============================================================
# DATA PREPARATION
# ============================================================

def prepare_ea(ea):
    ea = ea.copy()

    if "full_name" in ea.columns:
        ea["name_key"] = ea["full_name"].fillna(ea.get("name")).apply(normalize_name)
    else:
        ea["name_key"] = ea["name"].apply(normalize_name)

    ea["dob"] = safe_datetime(ea["dob"]) if "dob" in ea.columns else pd.NaT

    # Numeric conversions
    for col in ["value", "wage", "release_clause", "height_cm", "weight_kg", "overall_rating", "potential"]:
        if col in ea.columns:
            ea[col] = pd.to_numeric(ea[col], errors="coerce")

    # Date-based features
    ref_date = pd.Timestamp("2025-01-01")
    if "dob" in ea.columns:
        ea["age_ea"] = (ref_date - ea["dob"]).dt.days / 365.25
        ea["age_ea_sq"] = ea["age_ea"] ** 2

    if "club_contract_valid_until" in ea.columns:
        ea["club_contract_valid_until"] = safe_datetime(ea["club_contract_valid_until"])
        ea["contract_years_left_ea"] = ((ea["club_contract_valid_until"] - ref_date).dt.days / 365.25).clip(lower=0)

    if {"wage", "value"}.issubset(ea.columns):
        ea["wage_to_ea_value"] = ea["wage"] / ea["value"].replace(0, np.nan)

    ea = add_position_group(ea)
    return ea


def prepare_tm(tm):
    tm = tm.copy()

    tm["name_key"] = tm["name"].apply(normalize_name)
    if "date_of_birth" in tm.columns:
        tm["date_of_birth"] = safe_datetime(tm["date_of_birth"])
    if "date" in tm.columns:
        tm["date"] = safe_datetime(tm["date"])

    # Keep latest valuation if repeated
    if {"name_key", "date_of_birth", "date"}.issubset(tm.columns):
        tm = tm.sort_values(["name_key", "date_of_birth", "date"], ascending=[True, True, False])
        tm = tm.drop_duplicates(["name_key", "date_of_birth"], keep="first").copy()

    for col in ["market_value_in_eur_valuation", "contract_years_left", "age_at_valuation"]:
        if col in tm.columns:
            tm[col] = pd.to_numeric(tm[col], errors="coerce")

    if "age_at_valuation" in tm.columns:
        tm["age_at_valuation_sq"] = tm["age_at_valuation"] ** 2
        tm["abs_years_from_27"] = (tm["age_at_valuation"] - 27).abs()

    tm = add_position_group(tm)
    return tm


def prepare_fb(fb):
    fb = fb.copy()

    fb["name_key"] = fb["Player"].apply(normalize_name)
    fb = add_fbref_per90(fb)

    # Convert likely numeric columns
    possible_numeric = [
        "Age", "MP", "Starts", "Subs", "Min", "90s", "Gls", "Ast", "G+A",
        "G-PK", "PK", "PKatt", "Sh", "SoT", "Int", "TklW",
        "Gls/90", "Int/90", "Sh/90", "SoT/90", "TklW/90", "SoT%"
    ]
    for col in possible_numeric:
        if col in fb.columns:
            fb[col] = pd.to_numeric(fb[col], errors="coerce")

    # Keep one row per player, usually the highest minutes one
    if {"name_key", "Min"}.issubset(fb.columns):
        fb = fb.sort_values(["name_key", "Min"], ascending=[True, False])
        fb = fb.drop_duplicates("name_key", keep="first").copy()
    else:
        fb = fb.drop_duplicates("name_key", keep="first").copy()

    fb = add_position_group(fb)
    return fb


def add_target(df):
    """
    Build the final modelling target:
    average(EA value, TM value) where possible.
    """
    df = df.copy()

    ea_val = pd.to_numeric(df["value"], errors="coerce") if "value" in df.columns else np.nan
    tm_val = pd.to_numeric(df["market_value_in_eur_valuation"], errors="coerce") if "market_value_in_eur_valuation" in df.columns else np.nan

    if isinstance(ea_val, pd.Series) and isinstance(tm_val, pd.Series):
        df["target_value"] = np.where(
            ea_val.notna() & tm_val.notna(),
            (ea_val + tm_val) / 2.0,
            np.where(ea_val.notna(), ea_val, tm_val)
        )
    elif isinstance(tm_val, pd.Series):
        df["target_value"] = tm_val
    elif isinstance(ea_val, pd.Series):
        df["target_value"] = ea_val
    else:
        raise ValueError("No usable target value columns found.")

    df = df[df["target_value"].notna() & (df["target_value"] > 0)].copy()
    df["log_target"] = np.log1p(df["target_value"])

    return df


# ============================================================
# FEATURE LISTS
# ============================================================

def get_feature_sets(df):
    ea_features = get_existing_columns(df, [
        "position_group", "positions", "preferred_foot", "club_league_name", "country_name",
        "height_cm", "weight_kg", "overall_rating", "potential",
        "weak_foot", "skill_moves",
        "attacking_crossing", "attacking_finishing", "attacking_heading_accuracy",
        "attacking_short_passing", "attacking_volleys",
        "skill_dribbling", "skill_curve", "skill_fk_accuracy",
        "skill_long_passing", "skill_ball_control",
        "movement_acceleration", "movement_sprint_speed",
        "movement_agility", "movement_reactions", "movement_balance",
        "power_shot_power", "power_jumping", "power_stamina",
        "power_strength", "power_long_shots",
        "mentality_aggression", "mentality_interceptions",
        "mentality_vision", "mentality_penalties", "mentality_composure",
        "mentality_attack_position",
        "defending_defensive_awareness", "defending_standing_tackle", "defending_sliding_tackle",
        "goalkeeping_gk_diving", "goalkeeping_gk_handling", "goalkeeping_gk_kicking",
        "goalkeeping_gk_positioning", "goalkeeping_gk_reflexes",
        "age_ea", "age_ea_sq", "contract_years_left_ea"
    ])

    fb_features = get_existing_columns(df, [
        "position_group", "Pos", "Comp", "Age", "MP", "Starts", "Subs", "Min", "90s",
        "Gls", "Ast", "G+A", "G-PK", "PK", "PKatt", "Sh", "SoT", "Int", "TklW",
        "Gls/90", "Int/90", "Sh/90", "SoT/90", "TklW/90", "SoT%",
        "Gls_per90_calc", "Ast_per90_calc", "G+A_per90_calc", "G-PK_per90_calc",
        "PK_per90_calc", "PKatt_per90_calc", "Sh_per90_calc", "SoT_per90_calc",
        "Int_per90_calc", "TklW_per90_calc",
        "goal_contrib", "goal_contrib_per90", "start_ratio", "minutes_per_match"
    ])

    tm_features = get_existing_columns(df, [
        "position_group", "position", "sub_position",
        "player_club_domestic_competition_id",
        "contract_years_left", "age_at_valuation", "age_at_valuation_sq", "abs_years_from_27",
        "market_value_in_eur_valuation"
    ])

    meta_features = get_existing_columns(df, [
        "ea_pred", "fb_pred", "tm_pred",
        "position_group", "positions", "Pos", "position", "sub_position",
        "preferred_foot", "club_league_name", "country_name", "Comp",
        "value", "market_value_in_eur_valuation", "wage", "release_clause",
        "overall_rating", "potential",
        "height_cm", "weight_kg",
        "age_ea", "age_ea_sq",
        "age_at_valuation", "age_at_valuation_sq", "abs_years_from_27",
        "contract_years_left_ea", "contract_years_left",
        "wage_to_ea_value",
        "Min", "90s", "goal_contrib_per90", "Gls/90", "Int/90", "Sh/90", "SoT/90"
    ])

    return ea_features, fb_features, tm_features, meta_features


# ============================================================
# MAIN PIPELINE
# ============================================================

def main():
    print("Loading datasets...")
    ea = pd.read_csv(EA_PATH)
    tm = pd.read_csv(TM_PATH)
    fb = pd.read_csv(FB_PATH)

    print("Raw dataset shapes:")
    print("EA:", ea.shape)
    print("TM:", tm.shape)
    print("FB:", fb.shape)

    print("\nPreparing datasets...")
    ea = prepare_ea(ea)
    tm = prepare_tm(tm)
    fb = prepare_fb(fb)

    # --------------------------------------------------------
    # 1. EA-TM match on name + dob
    # --------------------------------------------------------
    print("\nMerging EA and Transfermarkt on name + dob...")
    ea_tm = ea.merge(
        tm,
        left_on=["name_key", "dob"],
        right_on=["name_key", "date_of_birth"],
        how="inner",
        suffixes=("_ea", "_tm")
    )

    # --------------------------------------------------------
    # 2. Join FBref by normalized name
    # --------------------------------------------------------
    print("Joining FBref by normalized name...")
    merged = ea_tm.merge(
        fb,
        on="name_key",
        how="left",
        suffixes=("", "_fb")
    )

    merged = add_position_group(merged)
    merged = add_target(merged)

    print("Merged shape:", merged.shape)
    print("Rows with FBref match:", merged["Player"].notna().sum() if "Player" in merged.columns else 0)

    # --------------------------------------------------------
    # Train / test split
    # --------------------------------------------------------
    train_df, test_df = train_test_split(
        merged,
        test_size=0.2,
        random_state=RANDOM_STATE
    )

    print("\nTrain shape:", train_df.shape)
    print("Test shape:", test_df.shape)

    # --------------------------------------------------------
    # Feature sets
    # --------------------------------------------------------
    ea_features, fb_features, tm_features, meta_features = get_feature_sets(merged)

    print("\nFeature counts:")
    print("EA features:", len(ea_features))
    print("FBref features:", len(fb_features))
    print("TM features:", len(tm_features))
    print("Meta features:", len(meta_features))

    # --------------------------------------------------------
    # Base model 1: EA
    # --------------------------------------------------------
    print("\nTraining EA base model...")
    ea_result = generate_oof_preds(
        train_df=train_df,
        test_df=test_df,
        feature_cols=ea_features,
        target_col="log_target",
        label="EA_BASE"
    )
    train_df["ea_pred"] = ea_result["oof"]
    test_df["ea_pred"] = ea_result["test_preds"]

    # --------------------------------------------------------
    # Base model 2: FBref
    # --------------------------------------------------------
    print("\nTraining FBref base model...")
    fb_result = generate_oof_preds(
        train_df=train_df,
        test_df=test_df,
        feature_cols=fb_features,
        target_col="log_target",
        label="FB_BASE"
    )
    train_df["fb_pred"] = fb_result["oof"]
    test_df["fb_pred"] = fb_result["test_preds"]

    # --------------------------------------------------------
    # Base model 3: Transfermarkt
    # --------------------------------------------------------
    print("\nTraining Transfermarkt base model...")
    tm_result = generate_oof_preds(
        train_df=train_df,
        test_df=test_df,
        feature_cols=tm_features,
        target_col="log_target",
        label="TM_BASE"
    )
    train_df["tm_pred"] = tm_result["oof"]
    test_df["tm_pred"] = tm_result["test_preds"]

    # --------------------------------------------------------
    # Meta model
    # --------------------------------------------------------
    print("\nTraining meta model...")
    meta_result = generate_oof_preds(
        train_df=train_df,
        test_df=test_df,
        feature_cols=meta_features,
        target_col="log_target",
        label="META"
    )

    # --------------------------------------------------------
    # Final predictions and evaluation
    # --------------------------------------------------------
    y_true = np.expm1(test_df["log_target"].values)
    y_pred = np.expm1(meta_result["test_preds"])

    metrics = regression_metrics(y_true, y_pred)

    print("\nFINAL METRICS")
    for k, v in metrics.items():
        print(f"{k}: {v:,.4f}")

    # --------------------------------------------------------
    # Save predictions
    # --------------------------------------------------------
    output_name_col = "full_name" if "full_name" in test_df.columns else "name"
    predictions = test_df[[output_name_col, "position_group"]].copy()
    predictions["actual_value"] = y_true
    predictions["predicted_value"] = y_pred
    predictions["abs_pct_error"] = np.abs((predictions["predicted_value"] - predictions["actual_value"]) / predictions["actual_value"]) * 100

    pred_path = os.path.join(OUTPUT_DIR, "predicted_player_values.csv")
    predictions.to_csv(pred_path, index=False)

    metrics_path = os.path.join(OUTPUT_DIR, "metrics.json")
    save_json(metrics, metrics_path)

    print("\nSaved:")
    print(pred_path)
    print(metrics_path)

    # --------------------------------------------------------
    # SHAP
    # --------------------------------------------------------
    if USE_SHAP:
        print("\nRunning SHAP on final meta model...")
        try:
            x_train_meta = meta_result["x_train"].copy()

            # Sample for speed if dataset is large
            if len(x_train_meta) > 500:
                x_sample = x_train_meta.sample(500, random_state=RANDOM_STATE)
            else:
                x_sample = x_train_meta

            explainer = shap.TreeExplainer(meta_result["model"])
            shap_values = explainer.shap_values(x_sample)

            mean_abs_shap = np.abs(shap_values).mean(axis=0)

            shap_df = pd.DataFrame({
                "feature": x_sample.columns,
                "mean_abs_shap": mean_abs_shap
            }).sort_values("mean_abs_shap", ascending=False)

            shap_path = os.path.join(OUTPUT_DIR, "shap_feature_importance.csv")
            shap_df.to_csv(shap_path, index=False)

            print("Top SHAP features:")
            print(shap_df.head(20).to_string(index=False))
            print("\nSaved:")
            print(shap_path)

        except Exception as e:
            print("\nSHAP could not be computed.")
            print("Reason:", str(e))

    print("\nDone.")


if __name__ == "__main__":
    main()

Loading datasets...
Raw dataset shapes:
EA: (17499, 57)
TM: (19637, 23)
FB: (10713, 31)

Preparing datasets...

Merging EA and Transfermarkt on name + dob...
Joining FBref by normalized name...
Merged shape: (6196, 135)
Rows with FBref match: 3657

Train shape: (4956, 135)
Test shape: (1240, 135)

Feature counts:
EA features: 47
FBref features: 37
TM features: 7
Meta features: 31

Training EA base model...
[EA_BASE] fold 1/5 complete
[EA_BASE] fold 2/5 complete
[EA_BASE] fold 3/5 complete
[EA_BASE] fold 4/5 complete
[EA_BASE] fold 5/5 complete

Training FBref base model...
[FB_BASE] fold 1/5 complete
[FB_BASE] fold 2/5 complete
[FB_BASE] fold 3/5 complete
[FB_BASE] fold 4/5 complete
[FB_BASE] fold 5/5 complete

Training Transfermarkt base model...
[TM_BASE] fold 1/5 complete
[TM_BASE] fold 2/5 complete
[TM_BASE] fold 3/5 complete
[TM_BASE] fold 4/5 complete
[TM_BASE] fold 5/5 complete

Training meta model...
[META] fold 1/5 complete
[META] fold 2/5 complete
[META] fold 3/5 complete
[ME